<a href="https://colab.research.google.com/github/jRicardo2003/etl-data-pipeline/blob/main/notebooks/seguros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd

url = "https://raw.githubusercontent.com/jRicardo2003/etl-data-pipeline/refs/heads/main/data/raw/aseguradoras.csv"
df = pd.read_csv(url)

df.head()

,id_aseguradora,nombre,pais,rating_riesgo
0,1,Aseguradora 1,Costa Rica,Alto
1,2,Aseguradora 2,El Salvador,Bajo
2,3,Aseguradora 3,El Salvador,NaN
3,4,Aseguradora 4,Costa Rica,Medio
4,5,Aseguradora 5,ElSalvador,B


In [2]:
df.shape
df.columns
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_aseguradora  15 non-null     int64 
 1   nombre          15 non-null     object
 2   pais            13 non-null     object
 3   rating_riesgo   12 non-null     object
dtypes: int64(1), object(3)
memory usage: 612.0+ bytes


,0
id_aseguradora,0
nombre,0
pais,2
rating_riesgo,3


**LIMPIEZA DE DATOS**

In [3]:
aseguradoras = df.copy()

# nombres de columnas
aseguradoras.columns = aseguradoras.columns.str.strip().str.lower()

# limpiar espacios en texto
for col in aseguradoras.select_dtypes(include='object').columns:
    aseguradoras[col] = aseguradoras[col].astype(str).str.strip()

# convertir vacíos en null
aseguradoras = aseguradoras.replace(r'^\s*$', pd.NA, regex=True)

# eliminar duplicados
aseguradoras = aseguradoras.drop_duplicates()

**CREACION DE CURATED Y REJECTS**

In [4]:
aseguradoras = df.copy()

# nombres de columnas
aseguradoras.columns = aseguradoras.columns.str.strip().str.lower()

# limpiar espacios en texto
for col in aseguradoras.select_dtypes(include='object').columns:
    aseguradoras[col] = aseguradoras[col].astype(str).str.strip()

# convertir vacíos en null
aseguradoras = aseguradoras.replace(r'^\s*$', pd.NA, regex=True)

# eliminar duplicados
aseguradoras = aseguradoras.drop_duplicates()

In [5]:

validos = aseguradoras[
    aseguradoras['nombre'].notna() &
    aseguradoras['pais'].notna()
].copy()

rechazados = aseguradoras[
    aseguradoras['nombre'].isna() |
    aseguradoras['pais'].isna()
].copy()


In [6]:
def motivo(row):
    motivos = []

    if pd.isna(row['nombre']):
        motivos.append("nombre_vacio")

    if pd.isna(row['pais']):
        motivos.append("pais_vacio")

    return ",".join(motivos)

rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)

In [7]:
validos.to_csv("aseguradoras_curated.csv", index=False)
rechazados.to_csv("aseguradoras_rejects.csv", index=False)